# Heart Disease Prediction - Decision Tree Classifier (Day 4)

This notebook covers the training, evaluation, tree visualization, and feature importance analysis of a Decision Tree Classifier trained on the preprocessed combined UCI Heart Disease dataset.

## Step 1: Import Libraries

**What & Why:**
- **What:** Imported data handling, modeling, plotting, and tree visualization modules.
- **Why:** These provide the classes required to instantiate a Decision Tree model, fit the model on raw features, compute evaluation metrics, and draw the actual conditional splits of the tree.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

## Step 2: Load Processed Dataset

**What & Why:**
- **What:** Loaded `heart_processed.csv` from the `data/` directory.
- **Why:** To fetch the cleaned, imputed, and encoded DataFrame for model training.

In [ ]:
df = pd.read_csv("../data/heart_processed.csv")
df.head()

## Step 3: Split Features and Target

**What & Why:**
- **What:** Partitioned the clean DataFrame into feature variables `X` and label `y`.
- **Why:** Separation of predictor attributes from the target is necessary for training inputs in scikit-learn.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

## Step 4: Train-Test Split

**What, Why & Observations:**
- **What:** Split the dataset into 80% training and 20% test data with stratification.
- **Why:** Stratified splitting ensures identical class distributions. Note that unlike Logistic Regression, **Decision Trees do not require feature scaling** because tree models split nodes based on thresholds of individual features, which are invariant to scale transformations.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 5: Create the Model

**What & Why:**
- **What:** Instantiated the `DecisionTreeClassifier` model.
- **Why:** We fix `random_state=42` to ensure consistent node splitting behavior during tree growth across evaluations.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)

## Step 6: Train the Model

**What & Why:**
- **What:** Fitted the Decision Tree Classifier on the training partition.
- **Why:** To optimize decision split thresholds based on information gain (Gini impurity or entropy reduction).

In [ ]:
tree.fit(X_train, y_train)

## Step 7: Make Predictions

**What & Why:**
- **What:** Ran predictions on the test feature split (`X_test`).
- **Why:** To generate predictions to compare against the true actual outcomes (`y_test`).

In [ ]:
y_pred = tree.predict(X_test)

## Step 8: Accuracy

**What, Why & Observations:**
- **What:** Computed test accuracy score.
- **Why:** To establish a base classification metric of model correctness.
- **Observations:** The Decision Tree Classifier achieved an accuracy of **76.09%**, which is lower than the **82.07%** achieved by Logistic Regression.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

## Step 9: Confusion Matrix

**What, Why & Observations:**
- **What:** Plotted the confusion matrix highlighting incorrect prediction groups.
- **Why:** To verify specific errors. In medical tasks, it is critical to keep False Negatives (FN) as low as possible.
- **Observations:** 
  - 58 patients were correctly predicted as healthy (True Negatives).
  - 82 patients were correctly predicted as diseased (True Positives).
  - 24 patients were incorrectly predicted as diseased (False Positives).
  - 20 patients were incorrectly predicted as healthy (False Negatives).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Greens")
plt.title("Decision Tree Confusion Matrix")
plt.show()

## Step 10: Classification Report

**What, Why & Observations:**
- **What:** Generated a classification report showing precision, recall, and F1-score.
- **Why:** To measure class-specific correctness and evaluate performance against Logistic Regression.
- **Observations:** 
  - **Class 0 (Healthy):** F1-score: **0.72** (Precision: 0.74, Recall: 0.71)
  - **Class 1 (Disease):** F1-score: **0.79** (Precision: 0.77, Recall: 0.80)
  - Compared to Logistic Regression (Class 0 F1-score: 0.80, Class 1 F1-score: 0.84), the Decision Tree model performs worse across all metrics.

In [ ]:
print(classification_report(y_test, y_pred))

## Step 11: Visualize the Decision Tree

**What, Why & Observations:**
- **What:** Plotted the decision boundary flowchart of the tree using `plot_tree`.
- **Why:** Visualizing splits explains how the model reaches decisions, showcasing root splits, Gini values, and leaf samples.
- **Observations:** The root split node uses chest pain (`cp`) as its first partition, followed by cholesterol (`chol`) and age. This confirms that clinical chest pain types represent the highest information gain feature.

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=["No Disease", "Disease"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()

## Step 12: Feature Importance

**What, Why & Observations:**
- **What:** Computed and displayed feature importances using the Gini reduction index.
- **Why:** To quantify the predictive contribution of each attribute.
- **Observations:** Chest pain (`cp`) is the most important feature (0.257), followed by cholesterol (`chol`, 0.200) and age (0.133). This explains why they are positioned at the top split nodes in the tree diagram.

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree.feature_importances_
})
importance = importance.sort_values(
    by="Importance",
    ascending=True
)
importance

In [ ]:
# Plot Feature Importance
plt.figure(figsize=(10,6))
plt.barh(
    importance["Feature"],
    importance["Importance"]
)
plt.title("Decision Tree Feature Importance")
plt.xlabel("Importance")
plt.show()

## Step 13: Prevent Overfitting

**What, Why & Observations:**
- **What:** Trained a pruned Decision Tree (`tree2`) limiting its depth to 4 (`max_depth=4`).
- **Why:** Fully grown decision trees split until all leaves are pure, memorizing training data (overfitting) and leading to poorer generalization. Pruning restricts growth, improving test generalization.
- **Observations:** Limiting depth to 4 increases the test set accuracy significantly from **76.09%** to **80.43%**, showing that pruning helps the model generalize better on unseen data.

In [ ]:
tree2 = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)
tree2.fit(X_train, y_train)
y_pred2 = tree2.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred2))